# Обозначения и единицы: regex vs NLP

Цель — понять, закрывает ли NLP (NER/эмбеддинги) задачу извлечения **обозначений и единиц** из нормативных формул лучше, чем текущий regex (`symbols.py` + `extract_formula_notation`), и что реально нужно для MVP.

Ключевая развилка: настоящий NER (token classification) требует fine-tune на размеченном корпусе, которого у нас нет. Проверяем, что даёт предобученный `rubert-tiny2` без дообучения.

In [ ]:
from __future__ import annotations

import sys
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import pdfplumber

# Путь к корню репозитория (надёжно при запуске из любого cwd).
ROOT = Path.cwd().resolve()
while not (ROOT / "docs" / "normative-base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))

from normatives_pipeline.parser.extractors import extract_formula_notation, extract_formulas

PDF = ROOT / "docs" / "normative-base" / "СП 22.13330.2016.pdf"

print("ROOT:", ROOT)


## Baseline: что извлекает regex-подход

`extract_formula_notation` ищет после формулы блок «где …» и строки вида `обозначение — расшифровка` (паттерн из конфига).

In [ ]:
with pdfplumber.open(PDF) as pdf:
    pages_text = [p.extract_text() or "" for p in pdf.pages[:60]]

formulas = extract_formulas(pages_text)
notation = extract_formula_notation(pages_text, formulas)
total = sum(len(v) for v in notation.values())
print(f"формул: {len(formulas)} | c расшифровкой обозначений: {len(notation)} | всего записей: {total}")
print()
for key in list(notation)[:5]:
    entries = notation[key]
    print(f"--- формула {key}: {[(e.symbol, e.meaning[:40]) for e in entries[:3]]}")


**Наблюдение:** regex-подход работает, но завязан на слово «где» и формат `символ — текст`. На сканах без текстового слоя, при других формулировках или табличных обозначениях он молчит. Проверим, что даёт NLP.

## rubert-tiny2 — это masked LM, а не NER

Модель `cointegrated/rubert-tiny2` обучена предсказывать маскированные токены (`BertForPreTraining`), а не размечать сущности. Показываем оба факта.

In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model="cointegrated/rubert-tiny2", device=-1)
print("fill-mask:", fill("[MASK] — удельный вес грунта")[0]["sequence"])
print()
print(
    "Архитектура rubert-tiny2 = BertForPreTraining (masked LM): "
    "NER/token-classification головы нет — размечать сущности напрямую она не умеет."
)


## Что даёт предобученная модель без fine-tune: эмбеддинги

Смотрим, группируют ли эмбеддинги варианты написания одного обозначения (`γII` vs `гамма_II`) и отделяют ли разные величины (`γII` vs `cII` vs `b`).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("cointegrated/rubert-tiny2")
texts = [
    "γII — удельный вес грунта",
    "гамма_II",
    "γII",
    "cII — удельное сцепление",
    "b — ширина подошвы фундамента",
]
embs = model.encode(texts, normalize_embeddings=True)


def cos(i: int, j: int) -> float:
    return float(np.dot(embs[i], embs[j]))


for i, j, label in [
    (0, 2, "γII — удельный вес грунта  vs  γII"),
    (2, 1, "γII  vs  гамма_II (тот же символ)"),
    (2, 3, "γII  vs  cII (разные величины)"),
    (2, 4, "γII  vs  b"),
]:
    print(f"{label:42s} {cos(i, j):.3f}")


## Выводы

- **Настоящий NER (token classification) требует fine-tune** на размеченном корпусе (например, NEREL или собственная разметка «обозначение/единица/величина»). Предобученный `rubert-tiny2` — это masked LM без NER-головы, «из коробки» он сущности не размечает.
- **Эмбеддинги без дообучения полезны для нормализации**: `γII` и `гамма_II` близки (0.85), то есть можно сгруппировать варианты написания одного обозначения. Но разные величины (`γII` vs `cII`) без контекста различаются слабо (0.77) — для надёжной классификации нужен либо fine-tune, либо правила.
- **Практичный план для MVP**: оставить regex (`symbols.py` + `extract_formula_notation`) для структурированных «где … символ — расшифровка», а эмбеддинги `rubert-tiny2` использовать как вспомогательный слой нормализации/дедупликации обозначений. Полноценный NER — отдельная задача с разметкой датасета, вне текущего MVP.
- Следующий шаг — сводный документ `recommendation` с метриками всех nb01–nb04 и выбором моделей для интеграции в `normatives_pipeline`.